# Cleaning of the final dataset

---

In [1]:
import duckdb
import pandas as pd

merged_full_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full.parquet"

output_file = merged_full_file.replace(".parquet", "_cleaned.parquet")
final_file = output_file.replace(".parquet", "_encoded.parquet")

In [2]:
# Connect to DuckDB
con = duckdb.connect()

In [3]:
# TODO remove
bookstate_query = f"""
WITH deduped AS (
    SELECT DISTINCT * FROM '{merged_full_file}'
)
SELECT book_state, COUNT(*) as count
FROM deduped
GROUP BY book_state
ORDER BY count DESC;
"""

bookstate_df = con.execute(bookstate_query).fetchdf()
print(bookstate_df)

   book_state      count
0           0  120490427


### Clean Data

In [3]:
# Compare book_state columns and remove duplicates + drop book_state_1, drop duplicate rows
comparison_query = f"""
WITH deduped AS (
    SELECT DISTINCT * FROM '{merged_full_file}'
)
SELECT
    SUM(CASE WHEN book_state = book_state_1 THEN 1 ELSE 0 END) AS matching,
    SUM(CASE WHEN book_state != book_state_1 THEN 1 ELSE 0 END) AS mismatching,
    COUNT(*) AS total_rows,
    ROUND(100.0 * SUM(CASE WHEN book_state = book_state_1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_matching,
    ROUND(100.0 * SUM(CASE WHEN book_state != book_state_1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mismatching
FROM deduped;
"""

In [4]:
# Execute the comparison separately to print stats
comparison_stats = con.execute(comparison_query).fetchdf()
print("Book State Comparison Summary:")
print(comparison_stats)

Book State Comparison Summary:
      matching  mismatching  total_rows  pct_matching  pct_mismatching
0  120490427.0          0.0   120490427         100.0              0.0


In [5]:
# Save the cleaned final dataset (deduped and without book_state_1)
copy_query = f"""
COPY (
    WITH deduped AS (
        SELECT DISTINCT * FROM '{merged_full_file}'
    )
    SELECT * EXCLUDE (book_state_1) FROM deduped
) TO '{output_file}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""
con.execute(copy_query)
print(f"✅ Cleaned dataset saved to: {output_file}")

✅ Cleaned dataset saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full_cleaned.parquet


In [6]:
con.close()

### Uniformly Encode Categorical Columns and Convert Dates to UNIX Timestamp

In [8]:
con = duckdb.connect()

In [9]:
schema = con.execute(f"DESCRIBE SELECT * FROM '{output_file}'").fetchdf()
print(schema)

                 column_name               column_type null   key default  \
0        measure_step_number                   INTEGER  YES  None    None   
1              measure_value                    DOUBLE  YES  None    None   
2                 book_state                   INTEGER  YES  None    None   
3   measurement_name_encoded                    BIGINT  YES  None    None   
4   measurement_unit_encoded                    BIGINT  YES  None    None   
5           is_within_limits                   INTEGER  YES  None    None   
6        workstep_number_mes                   INTEGER  YES  None    None   
7                 book_stamp  TIMESTAMP WITH TIME ZONE  YES  None    None   
8                 part_group                   VARCHAR  YES  None    None   
9           serial_number_id                   VARCHAR  YES  None    None   
10                station_id                   VARCHAR  YES  None    None   
11        component_position                   VARCHAR  YES  None    None   

In [14]:
# Encode using hash as numeric float (to match float-only dataset requirement)
con.execute(f"""
COPY (
    SELECT
        measure_step_number,
        measure_value,
        is_within_limits,
        workstep_number_mes,
        book_state,
        container_number_freq,
        EXTRACT(EPOCH FROM book_stamp) AS book_stamp_unix,
        CAST(HASH(measurement_name_encoded) AS DOUBLE) AS measurement_name_encoded_enc,
        CAST(HASH(measurement_unit_encoded) AS DOUBLE) AS measurement_unit_encoded_enc,
        CAST(HASH(part_group) AS DOUBLE) AS part_group_enc,
        CAST(HASH(serial_number_id) AS DOUBLE) AS serial_number_id_enc,
        CAST(HASH(station_id) AS DOUBLE) AS station_id_enc,
        CAST(HASH(component_position) AS DOUBLE) AS component_position_enc,
        CAST(HASH(component_id) AS DOUBLE) AS component_id_enc,
        CAST(HASH(panel_position) AS DOUBLE) AS panel_position_enc,
        CAST(HASH(supplier_id) AS DOUBLE) AS supplier_id_enc,
        CAST(HASH(mounting_place) AS DOUBLE) AS mounting_place_enc,
    FROM '{output_file}'
) TO '{final_file}' (FORMAT PARQUET, COMPRESSION 'zstd');
""")

print(f"Categorical columns encoded to floats using HASH. Final dataset saved to: {final_file}")

Categorical columns encoded to floats using HASH. Final dataset saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full_cleaned_timestamped_encoded.parquet


In [15]:
schema = con.execute(f"DESCRIBE SELECT * FROM '{final_file}'").fetchdf()
print(schema)

                     column_name column_type null   key default extra
0            measure_step_number     INTEGER  YES  None    None  None
1                  measure_value      DOUBLE  YES  None    None  None
2               is_within_limits     INTEGER  YES  None    None  None
3            workstep_number_mes     INTEGER  YES  None    None  None
4                     book_state     INTEGER  YES  None    None  None
5          container_number_freq      BIGINT  YES  None    None  None
6                book_stamp_unix      DOUBLE  YES  None    None  None
7   measurement_name_encoded_enc      DOUBLE  YES  None    None  None
8   measurement_unit_encoded_enc      DOUBLE  YES  None    None  None
9                 part_group_enc      DOUBLE  YES  None    None  None
10          serial_number_id_enc      DOUBLE  YES  None    None  None
11                station_id_enc      DOUBLE  YES  None    None  None
12        component_position_enc      DOUBLE  YES  None    None  None
13              comp

### Check cardinality of key categorical features

In [16]:
encoded_columns = [
    "measurement_name_encoded_enc",
    "measurement_unit_encoded_enc",
    "part_group_enc",
    "serial_number_id_enc",
    "station_id_enc",
    "component_position_enc",
    "component_id_enc",
    "panel_position_enc",
    "supplier_id_enc",
    "mounting_place_enc",
    "container_number_freq"
]

print("Cardinality of encoded categorical features in the final file:")
for col in encoded_columns:
    cardinality = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM '{final_file}'").fetchone()[0]
    print(f"   → {col}: {cardinality}")

Cardinality of encoded categorical features in the final file:
   → measurement_name_encoded_enc: 6
   → measurement_unit_encoded_enc: 4
   → part_group_enc: 2
   → serial_number_id_enc: 18708
   → station_id_enc: 2
   → component_position_enc: 368
   → component_id_enc: 144
   → panel_position_enc: 2
   → supplier_id_enc: 30
   → mounting_place_enc: 536
   → container_number_freq: 1434


### Compute FULL Pairwise Correlation Matrix for numeric columns

In [17]:
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{final_file}'").fetchdf()
numeric_cols = schema_df[schema_df['column_type'].str.contains('INT|DOUBLE|FLOAT|BIGINT')]['column_name'].tolist()

pairs = [(c1, c2) for i, c1 in enumerate(numeric_cols) for j, c2 in enumerate(numeric_cols) if i < j]
results = []
for c1, c2 in pairs:
    try:
        cor = con.execute(f"SELECT corr({c1}, {c2}) FROM '{final_file}'").fetchone()[0]
        results.append((c1, c2, cor))
    except Exception as e:
        print(f"Failed correlation for {c1} and {c2}: {e}")
        continue

correlation_df = pd.DataFrame(results, columns=["column_1", "column_2", "correlation"])
correlation_df.sort_values(by="correlation", ascending=False, inplace=True)

print("Top 20 most correlated numeric columns:")
print(correlation_df.head(20))

Top 20 most correlated numeric columns:
                         column_1                      column_2  correlation
52            workstep_number_mes                station_id_enc     1.000000
49            workstep_number_mes  measurement_unit_encoded_enc     0.725090
102  measurement_unit_encoded_enc                station_id_enc     0.725090
2             measure_step_number           workstep_number_mes     0.334076
10            measure_step_number                station_id_enc     0.334076
91   measurement_name_encoded_enc  measurement_unit_encoded_enc     0.236079
7             measure_step_number  measurement_unit_encoded_enc     0.175326
17                  measure_value           workstep_number_mes     0.172607
25                  measure_value                station_id_enc     0.172607
48            workstep_number_mes  measurement_name_encoded_enc     0.154258
94   measurement_name_encoded_enc                station_id_enc     0.154258
80          container_number_freq   

In [18]:
# Check distinct counts
result = con.execute(f"""
SELECT
    COUNT(DISTINCT workstep_number_mes) AS wsteps,
    COUNT(DISTINCT station_id_enc) AS stations
FROM '{final_file}';
""").fetchdf()

print(result)

   wsteps  stations
0       2         2


In [19]:
con.execute(f"""
COPY (
    SELECT * EXCLUDE (station_id_enc)
    FROM '{final_file}'
) TO '{final_file}' (FORMAT PARQUET, COMPRESSION 'zstd');
""")

print(f"Saved dataset without station_id_enc to: {final_file}")

Saved dataset without station_id_enc to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_full_cleaned_timestamped_encoded.parquet


In [20]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
